# 03 · Train ResNet-50

Full fine-tuning of **ResNet-50** (ImageNet-V2 pretrained) for 16-class classification.

**Checklist:**
- [ ] Notebook 02 complete — `/content/data` has train/val/test  OR  Drive split exists
- [ ] Runtime → Runtime type → **A100 GPU**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────
import os, sys

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ NOT FOUND')
print('CUDA:', torch.version.cuda)

In [ ]:
# ── Cell 2: Paths ─────────────────────────────────────────────────────
LOCAL_DATA_DIR   = '/content/data'
DRIVE_SPLIT_PATH = '/content/drive/MyDrive/pk_politicians_split'
DRIVE_RESULTS    = '/content/drive/MyDrive/pk_politicians_results'

In [ ]:
# ── Cell 3: Load data from Drive → SSD ────────────────────────────────
from src.utils import copy_dataset_from_drive, set_seed
from src.dataset import get_dataloaders

set_seed(42)
copy_dataset_from_drive(DRIVE_SPLIT_PATH, LOCAL_DATA_DIR)

loaders = get_dataloaders(
    train_dir = os.path.join(LOCAL_DATA_DIR, 'train'),
    val_dir   = os.path.join(LOCAL_DATA_DIR, 'val'),
    test_dir  = os.path.join(LOCAL_DATA_DIR, 'test'),
)

In [ ]:
# ── Cell 4: Build ResNet-50 ────────────────────────────────────────────
from src.models import build_model

model = build_model(
    model_name  = 'resnet50',
    num_classes = 16,
    dropout     = 0.4,
    freeze_base = False,   # full fine-tuning
)

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────
from src.train import Trainer

trainer = Trainer(
    model      = model,
    loaders    = loaders,
    model_name = 'resnet50',
    config_overrides = {
        'num_epochs':    30,
        'learning_rate': 1e-4,
        'weight_decay':  1e-4,
    }
)
history = trainer.run()

In [ ]:
# ── Cell 6: Training curves ────────────────────────────────────────────
from src.evaluate import plot_training_curves
plot_training_curves(history, model_name='resnet50').show()

In [ ]:
# ── Cell 7: Test set evaluation ───────────────────────────────────────
import torch
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_misclassified

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = evaluate_model(model, loaders['test'], device, model_name='resnet50')

plot_confusion_matrix(results['y_true'], results['y_pred'], model_name='resnet50').show()
fig = plot_misclassified(results['images'], results['y_true'], results['y_pred'], model_name='resnet50')
if fig: fig.show()

In [ ]:
# ── Cell 8: Log experiment ────────────────────────────────────────────
from src.utils import ExperimentLogger
logger = ExperimentLogger()
logger.log(
    model_name    = 'resnet50',
    test_accuracy = round(results['accuracy'], 4),
    macro_f1      = round(results['report']['macro avg']['f1-score'], 4),
    epochs_run    = len(history['train_loss']),
    lr=1e-4, batch_size=32,
    notes = 'Full fine-tuning, dropout=0.4, early stop patience=8',
)

In [ ]:
# ── Cell 9: Save results to Drive ─────────────────────────────────────
from src.utils import save_results_to_drive
save_results_to_drive('results', DRIVE_RESULTS)
print('✓ Saved. Next → 04_train_efficientnet.ipynb')